# CREAM Masking and Representation Analysis

This notebook discovers mask_debug_tensors.pt files under experiments/ and compares C--C density/topology runs, including standard, fixed-U2U, and fixed-U2U+U2C conditions. For each aligned test row it shows the extended concept channels entering the C--C layer, a dense no-mask U2C projection, actual masked-layer logits, and activated concept scores. Run-level test task/concept accuracy is loaded from capture metadata.

Run jobs with debug_masking.enabled: true to create captures. The test-row number is the sequential position in the test loader, so variants must use the same dataset split and loader ordering for row alignment. Existing text logs do not contain tensors and cannot be backfilled; rerun those jobs to create captures.

In [ ]:
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').exists() and (PROJECT_ROOT.parent / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CAPTURE_FILES = sorted((PROJECT_ROOT / 'experiments').rglob('mask_debug_tensors.pt'))

def read_capture(path):
    return torch.load(path, map_location='cpu', weights_only=False)

CAPTURES = []
for capture_path in CAPTURE_FILES:
    payload = read_capture(capture_path)
    meta = payload.get('metadata', {})
    for batch in payload.get('records', []):
        CAPTURES.append({
            'path': capture_path, 'metadata': meta, 'graph': payload.get('graph', {}),
            'test_row_indices': batch['test_row_indices'], 'tensors': batch['tensors'],
            'targets': batch.get('targets'),
        })

catalog_rows = []
for item in CAPTURES:
    m = item['metadata']
    catalog_rows.append({
        'dataset': m.get('dataset'), 'family': m.get('family'), 'variant': m.get('variant'),
        'condition': m.get('condition', 'unclassified'),
        'seed': m.get('seed'), 'experiment': m.get('experiment_name'),
        'first_test_row': int(item['test_row_indices'][0]),
        'last_test_row': int(item['test_row_indices'][-1]), 'capture_file': str(item['path']),
        'test_task_accuracy': m.get('test_metrics', {}).get('test_task_accuracy'),
        'test_concept_accuracy': m.get('test_metrics', {}).get('test_concept_accuracy'),
    })
catalog = pd.DataFrame(catalog_rows)
print(f'Found {len(CAPTURE_FILES)} capture files and {len(CAPTURES)} captured test batches.')
display(catalog.drop_duplicates('capture_file').sort_values(['dataset', 'family', 'condition', 'variant'], na_position='last') if not catalog.empty else catalog)

## C--C Experiment Coverage

The table below includes both C--C families and all available training conditions. A capture is one run; tensor batches are collapsed here so each run appears once. Missing families/conditions have no saved tensor capture yet and need to be rerun with debug tensor capture enabled.

In [ ]:
C2C_FAMILIES = ['concept_concept_density', 'concept_concept_fixcount_topology']
C2C_CONDITIONS = ['standard', 'fixed_u2u', 'fixed_u2u_fixed_u2c']

if catalog.empty:
    c2c_runs = catalog.copy()
else:
    c2c_runs = (catalog[catalog['family'].isin(C2C_FAMILIES)]
                .drop_duplicates('capture_file')
                .sort_values(['dataset', 'family', 'condition', 'variant']))
display(c2c_runs[['dataset', 'family', 'condition', 'variant', 'seed',
                  'test_task_accuracy', 'test_concept_accuracy', 'capture_file']]
        if not c2c_runs.empty else c2c_runs)

# Quick per-dataset/family/condition coverage summary.
if not c2c_runs.empty:
    display(c2c_runs.groupby(['dataset', 'family', 'condition'], dropna=False)
            .agg(runs=('capture_file', 'nunique'),
                 mean_test_task_accuracy=('test_task_accuracy', 'mean'),
                 mean_test_concept_accuracy=('test_concept_accuracy', 'mean'))
            .round(4))

## Inspect One C--C Family and Condition

Choose either C--C density or C--C fixed-count topology, and compare standard, fixed-U2U, or fixed-U2U+U2C runs. The coverage table above lists every C--C run found on disk.

In [ ]:
DATASET = 'Complete_Concept_FMNIST'  # e.g. CUB, CelebA
FAMILY = 'concept_concept_fixcount_topology'  # or concept_concept_density
CONDITION = 'fixed_u2u_fixed_u2c'  # also: fixed_u2u, standard
TEST_ROW = 0
VARIANTS = None  # or ['replace_020', 'replace_100']

def sample_record(item, row_index):
    rows = item['test_row_indices']
    matches = torch.where(rows == int(row_index))[0]
    if not len(matches):
        return None
    local = int(matches[0])
    values = {name: value[local].float().flatten() for name, value in item['tensors'].items()}
    for name, value in (item.get('targets') or {}).items():
        values[name] = value[local].float().flatten()
    return values

def selected_records(dataset, family, row_index, variants=None, condition=None):
    found = {}
    for item in CAPTURES:
        meta = item['metadata']
        if meta.get('dataset') != dataset or meta.get('family') != family:
            continue
        if condition is not None and meta.get('condition', 'unclassified') != condition:
            continue
        variant = str(meta.get('variant') or meta.get('experiment_name') or item['path'].parent.name)
        if variants is not None and variant not in variants:
            continue
        values = sample_record(item, row_index)
        if values is not None:
            seed = meta.get('seed')
            display_variant = f'{variant} (seed={seed})' if seed is not None else variant
            found[display_variant] = (values, item)
    return found

def concept_labels(item, count):
    names = item['metadata'].get('node_names') or []
    labels = [str(x) for x in names[:count]]
    return labels + [f'concept_{i}' for i in range(len(labels), count)]

def compare_sample(dataset=DATASET, family=FAMILY, row_index=TEST_ROW, variants=VARIANTS, condition=CONDITION):
    records = selected_records(dataset, family, row_index, variants, condition)
    if not records:
        print('No aligned captures found. Check dataset/family names and TEST_ROW range.')
        return
    names = sorted(records)
    print(f'Reference variant for delta/cosine columns: {names[0]}')
    reference = records[names[0]][0]
    stages = ['backbone_features', 'u2u_output', 'uc_before_c2c', 'concept_logits_unmasked', 'concept_scores_unmasked', 'concept_logits', 'concept_scores', 'task_logits']
    rows = []
    for variant in names:
        values = records[variant][0]
        for stage in stages:
            if stage not in values or stage not in reference or values[stage].shape != reference[stage].shape:
                continue
            a, b = values[stage], reference[stage]
            cosine = torch.nn.functional.cosine_similarity(a[None], b[None]).item()
            rows.append({'variant': variant, 'stage': stage, 'cosine_to_reference': cosine,
                         'mean_abs_delta': (a-b).abs().mean().item(), 'rmse': torch.mean((a-b)**2).sqrt().item()})
    scores = pd.DataFrame(rows)
    display(scores.pivot(index='stage', columns='variant', values='mean_abs_delta').round(5))
    fig, ax = plt.subplots(figsize=(max(8, len(names)*1.0), 3.8))
    sns.heatmap(scores.pivot(index='stage', columns='variant', values='cosine_to_reference'),
                vmin=-1, vmax=1, cmap='vlag', annot=True, fmt='.3f', ax=ax)
    ax.set_title(f'{dataset} / {family} / {condition} / test row {row_index}: representation similarity')
    plt.tight_layout(); plt.show()

    concept_names = concept_labels(records[names[0]][1], len(reference['concept_scores']))
    if 'true_concepts' in reference:
        display(pd.DataFrame([reference['true_concepts'].numpy()], columns=concept_names,
                             index=['ground-truth concepts for this test row']).round(3))
    for stage, title in [('concept_logits_unmasked', 'Dense counterfactual logits before applying C--C mask'),
                         ('concept_scores_unmasked', 'Activated counterfactual scores before applying C--C mask'),
                         ('concept_logits', 'Actual logits after the masked U2C layer'),
                         ('concept_scores', 'Activated final concept scores after the C--C mask')]:
        if not all(stage in records[v][0] for v in names):
            print(f'{stage} is absent from one or more captures; rerun those jobs with the updated tensor-capture code.')
            continue
        matrix = pd.DataFrame({v: records[v][0][stage].numpy() for v in names}, index=concept_names).T
        display(matrix.round(4).style.background_gradient(cmap='YlGnBu', axis=None))
        fig, ax = plt.subplots(figsize=(max(8, len(concept_names)*0.55), max(3.5, len(names)*0.45)))
        lo, hi = float(matrix.min().min()), float(matrix.max().max())
        bounds = {'vmin': 0, 'vmax': 1} if stage == 'concept_scores' and lo >= 0 and hi <= 1 else {}
        sns.heatmap(matrix, cmap='mako', annot=len(concept_names) <= 24, fmt='.2f', ax=ax, **bounds)
        ax.set_title(title + ' (same test example)')
        ax.set_xlabel('Concept'); ax.set_ylabel('Graph variant')
        plt.xticks(rotation=55, ha='right'); plt.tight_layout(); plt.show()

    metric_rows = []
    for variant in names:
        meta = records[variant][1]['metadata']
        metric_rows.append({'variant': variant, **meta.get('test_metrics', {})})
    display(pd.DataFrame(metric_rows).set_index('variant'))

    prediction_rows = []
    for variant in names:
        values = records[variant][0]
        if 'true_class' not in values:
            continue
        logits = values['task_logits']
        true_class = int(values['true_class'][0].item())
        if logits.numel() == 1:
            pred_class = int(logits.item() >= 0)
        else:
            pred_class = int(logits.argmax().item())
        prediction_rows.append({'variant': variant, 'true_class': true_class,
                               'predicted_class': pred_class, 'correct': pred_class == true_class})
    if prediction_rows:
        display(pd.DataFrame(prediction_rows).set_index('variant'))

    block_count, dims = records[names[0]][0]['uc_extended_blocks'].shape
    show_names = names[:min(4, len(names))]
    fig, axes = plt.subplots(len(show_names), 1, figsize=(max(8, dims*0.35), 2.3*len(show_names)), squeeze=False)
    for ax, variant in zip(axes.flat, show_names):
        arr = records[variant][0]['uc_extended_blocks'].reshape(block_count, dims).numpy()
        sns.heatmap(arr, cmap='rocket', ax=ax, cbar=ax is axes.flat[0],
                    yticklabels=concept_names, xticklabels=list(range(dims)))
        ax.set_title(f'{variant}: Uc before C--C masking; rows=source concepts, columns=extended channels')
        ax.set_xlabel('Channel within source-concept block'); ax.set_ylabel('Source concept')
    plt.tight_layout(); plt.show()

    for variant in show_names:
        graph = records[variant][1]['graph']
        mask = graph.get('expanded_u2c_mask')
        if mask is None:
            continue
        mask = mask.bool().numpy()
        fig, ax = plt.subplots(figsize=(max(8, mask.shape[1]*0.18), max(3.5, mask.shape[0]*0.28)))
        sns.heatmap(mask.astype(int), cmap=['#f0f0f0', '#2878b5'], cbar=False,
                    yticklabels=concept_names, xticklabels=False, ax=ax)
        input_block_width = mask.shape[1] // max(1, mask.shape[0])
        for block in range(1, mask.shape[0]):
            ax.axvline(block * input_block_width, color='#111111', linewidth=0.65, alpha=0.7)
        centers = np.arange(mask.shape[0]) * input_block_width + input_block_width / 2
        ax.set_xticks(centers); ax.set_xticklabels(concept_names, rotation=55, ha='right')
        ax.set_title(f'{variant}: expanded C--C mask (1=allowed channel, 0=blocked)')
        ax.set_xlabel(f'Input source-concept blocks (each block has {input_block_width} numbered channels)')
        ax.set_ylabel('Output concept')
        plt.tight_layout(); plt.show()
        for layer_name, layer in graph.get('u2c_layers', {}).items():
            if 'effective_weight' not in layer:
                continue
            effective = layer['effective_weight'].float()
            layer_mask = layer.get('mask', torch.ones_like(effective, dtype=torch.bool)).bool()
            blocked_max = effective[~layer_mask].abs().max().item() if (~layer_mask).any() else 0.0
            print(f'{variant} / U2C layer {layer_name}: active={int(layer_mask.sum())}/{layer_mask.numel()}, blocked effective max abs={blocked_max:.3g}')
            fig, ax = plt.subplots(figsize=(max(8, effective.shape[1]*0.18), max(3.5, effective.shape[0]*0.28)))
            sns.heatmap(effective.numpy(), cmap='vlag', center=0, xticklabels=False,
                        yticklabels=concept_names, ax=ax)
            ax.set_title(f'{variant}: learned effective U2C weights, layer {layer_name}')
            ax.set_xlabel('Extended input channel'); ax.set_ylabel('Output concept')
            plt.tight_layout(); plt.show()
            break

compare_sample()

## Compare Every C--C Capture for the Same Test Row

This overview aligns every available C--C density/topology capture for the selected dataset and test row, across standard, fixed-U2U, and fixed-U2U+U2C runs. It prints the four concept-vector stages and per-example task prediction together.

In [ ]:
def compare_all_c2c_for_sample(dataset=DATASET, row_index=TEST_ROW):
    runs = []
    for item in CAPTURES:
        meta = item['metadata']
        if meta.get('dataset') != dataset or meta.get('family') not in C2C_FAMILIES:
            continue
        values = sample_record(item, row_index)
        if values is None:
            continue
        variant = str(meta.get('variant') or meta.get('experiment_name') or item['path'].parent.name)
        run_name = f"{meta.get('family')} | {meta.get('condition', 'unclassified')} | {variant} | seed={meta.get('seed')}"
        runs.append((run_name, values, item))
    if not runs:
        print(f'No C--C tensor captures for {dataset}, test row {row_index}.')
        return
    runs.sort(key=lambda x: x[0])
    concept_names = concept_labels(runs[0][2], len(runs[0][1]['concept_scores']))
    for stage, title in [('concept_logits_unmasked', 'Pre-mask dense counterfactual logits'),
                         ('concept_scores_unmasked', 'Pre-mask activated counterfactual scores'),
                         ('concept_logits', 'Post-mask concept logits'),
                         ('concept_scores', 'Post-mask activated concept scores')]:
        available = [(name, values[stage]) for name, values, _ in runs if stage in values]
        if not available:
            print(f'{stage}: no saved values; rerun jobs with updated debug capture.')
            continue
        matrix = pd.DataFrame({name: tensor.numpy() for name, tensor in available}, index=concept_names).T
        print(title)
        display(matrix.round(4).style.background_gradient(cmap='YlGnBu', axis=None))
    rows = []
    for name, values, item in runs:
        meta = item['metadata']
        row = {'run': name, 'test_task_accuracy': meta.get('test_metrics', {}).get('test_task_accuracy'),
               'test_concept_accuracy': meta.get('test_metrics', {}).get('test_concept_accuracy')}
        if 'true_class' in values:
            logits = values['task_logits']
            row['true_class'] = int(values['true_class'][0].item())
            row['predicted_class'] = int(logits.item() >= 0) if logits.numel() == 1 else int(logits.argmax().item())
            row['correct_on_this_row'] = row['true_class'] == row['predicted_class']
        rows.append(row)
    display(pd.DataFrame(rows).set_index('run'))

compare_all_c2c_for_sample()

## What to Read
- Backbone features should match across runs if the same frozen backbone/cache and test example were used.
- U2U output should match when the shared U2U weights and backbone input match.
- uc_before_c2c is the extended channel representation before the masked C--C layer.
- The expanded mask marks which channels each output concept may use. Saved effective weights are raw learned weights multiplied by this mask, so blocked entries should be exactly zero.
- concept_logits_unmasked and concept_scores_unmasked are counterfactual dense projections/activations of the same extended input using the learned raw U2C weights with the mask bypassed; they are not a separately trained unmasked model.
- concept_logits are the actual output immediately after the masked concept layer and before activation; concept_scores are after activation.
- Compare these stages concept-by-concept: if dense logits differ but masked logits/scores converge, inspect the masks/effective weights. If scores are similar but task accuracy changes, compare task logits and the run-level test metrics.
- Different masks can still yield similar scores if allowed weights are near zero or allowed channels are redundant.